# 03 · Embeddings

**Goal:** understand what an embedding is, that it's just a lookup table, how meaning can emerge from it, and why it's scaled by `√d_model`.

Concepts: *embedding table*, *one-hot*, *cosine similarity*, *distributional meaning*, *scaling*.

Standalone — needs `numpy` and `matplotlib`.

## Words you'll meet
- **Token**: one piece of text the model reads, like one word. Think of a single Lego brick in a word spelled out with bricks.
- **Embedding**: a list of numbers that describes a token, like GPS coordinates describe a place. Tokens with similar meaning get nearby coordinates.
- **Vector**: just a list of numbers, such as `[0.1, -0.5, 2.0]`. One row of a table.
- **Embedding table**: a big table with one row for each token id, like a library catalogue where each book (identified by its shelf number, the id) has one card (row) of numbers.
- **One-hot**: a row of zeros with a single 1, like a class register where only one name is ticked.
- **Cosine similarity**: a score for "do these two arrows point the same way?" 1 means same direction, 0 means unrelated.
- **Co-occurrence**: counting which words show up near each other, like noticing which students always sit together.

## 1. Why not just use the id?
Token id 7 isn't "bigger" than id 3 in any meaningful way. We need a representation where **similar tokens are close**.
An **embedding** maps each token id to a learned vector of length `d_model`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

V, d_model = 6, 4
rng = np.random.default_rng(0)
E = rng.normal(size=(V, d_model))     # the embedding table, shape (V, d_model)
print(E.round(2))

> 🧒 **In plain words:**
> We make a table `E` with 6 rows (one per token) and 4 columns (4 numbers describing each token). The numbers are random, drawn from a bell-curve generator; `rng` is the random generator and the `0` is its starting seed, so you get the same numbers every time.
> `.round(2)` just shortens each number to 2 decimals for printing.
> **Notice:** 6 rows, 4 numbers each. These numbers mean nothing yet; they are like a blank catalogue that training will fill in.
> **Try it:** change `V, d_model = 6, 4` to `3, 2` and run again. You should see a smaller table.

## 2. An embedding is a table lookup
Row `i` of `E` is the vector for the token whose **id** is `i`. The table never sees the token itself ("cat"), only its id; the vocabulary (notebook 01) is what links the two:

`token --(vocabulary)--> token id --(row of E)--> embedding vector`

For a batch `(B, T)` of ids, indexing returns `(B, T, d_model)`.

In [ ]:
ids = np.array([[0, 3, 3, 5],
                [2, 1, 4, 4]])
x = E[ids]
print("ids", ids.shape, "->", x.shape)
print("row for token id 3:", E[3].round(2))
print("same at (0,1) :", x[0, 1].round(2))
print("same at (0,2) :", x[0, 2].round(2), " <- same token => same vector (no position info yet!)")

> 🧒 **In plain words:**
> `ids` is two sentences of 4 tokens each, written as token numbers. The shape `(2, 4)` means 2 rows (examples) by 4 columns (positions).
> `E[ids]` looks up the matching row of `E` for every number, like fetching a catalogue card for each book number. The result has shape `(2, 4, 4)`: 2 sentences, 4 positions, 4 numbers per token.
> **Notice:** token id 3 appears twice in the first sentence (positions 1 and 2), and both times we get exactly the same four numbers. The table does not know where in the sentence a word is.
> **Try it:** change a 3 in the first row to a 0. Position 1 or 2 will then show the vector of token 0 instead.

## 3. Equivalent view: one-hot × matrix
A lookup is the same as multiplying a **one-hot** vector by `E`. This is why the embedding is "a linear layer with no bias whose input is one-hot", and why gradients only update the rows that were used.

In [ ]:
one_hot = np.eye(V)[ids]                 # (B, T, V)
x2 = one_hot @ E                         # (B, T, V) @ (V, d) -> (B, T, d)
print("one_hot shape:", one_hot.shape)
print("identical to lookup:", np.allclose(x, x2))

> 🧒 **In plain words:**
> `np.eye(V)` makes a grid with 1s down the diagonal and 0s elsewhere; each row of it is a one-hot row (a single 1 at a different place). Indexing it with `ids` gives one such row per token, so the shape is `(2, 4, 6)`: 2 sentences, 4 positions, 6 slots each.
> The `@` sign is matrix multiplication: each one-hot row picks out exactly one row of `E`, because the 1 multiplies that row and all the 0s wipe out the rest.
> **Notice:** `np.allclose` prints `True`, meaning both methods give the same numbers. Picking a row and multiplying by a one-hot row are the same thing.

## 4. Random embeddings carry no meaning — training gives them meaning
At initialization, "cat" and "dog" are as far apart as "cat" and "banana". During training, gradient updates pull useful tokens together.

We can *see* this principle without training a neural net: the **distributional hypothesis** — *words used in similar contexts have similar meanings*. We'll build embeddings from co-occurrence statistics of a toy corpus.

In [ ]:
rng = np.random.default_rng(42)
animals = ["cat", "dog", "mouse", "horse"]
foods   = ["bread", "rice", "apple", "cheese"]
a_verbs = ["runs", "sleeps", "jumps"]
people_verbs = ["eats", "likes", "buys"]

sentences = []
for _ in range(600):
    if rng.random() < 0.5:
        sentences.append(["the", rng.choice(animals), rng.choice(a_verbs), "."])
    else:
        sentences.append(["she", rng.choice(people_verbs), "the", rng.choice(foods), "."])

for s in sentences[:5]:
    print(" ".join(s))

> 🧒 **In plain words:**
> We invent a tiny made-up language. Animal sentences look like "the cat runs ." and food sentences like "she eats the apple ." The loop writes 600 sentences, flipping a coin each time to pick which kind. `rng.choice` picks a random word from a list.
> **Notice:** the 5 printed sentences are a mix of both kinds. The model will never be told that cats are animals; it only sees the sentences.
> **Try it:** add `"owl"` to the `animals` list and re-run; "owl" will start appearing in sentences.

In [ ]:
vocab = sorted({w for s in sentences for w in s})
idx = {w: i for i, w in enumerate(vocab)}
V = len(vocab)

# count how often word j appears within +-2 positions of word i
C = np.zeros((V, V))
for s in sentences:
    for i, w in enumerate(s):
        for j in range(max(0, i - 2), min(len(s), i + 3)):
            if j != i:
                C[idx[w], idx[s[j]]] += 1

print("co-occurrence matrix", C.shape)

> 🧒 **In plain words:**
> First we list every different word (`vocab`) and give each a number (`idx`), like student ID numbers. Then we build `C`, a square scoreboard with one row and one column per word.
> The loops slide through each sentence; for each word we look at neighbours up to 2 places to the left or right (`range(max(0, i-2), min(len(s), i+3))` is just "stay inside the sentence"), and add 1 to the scoreboard cell for that pair.
> **Notice:** the shape is `(17, 17)`: there are 17 different words, and every word has a count against every other word.
> Think of it as counting who sits next to whom in class.

Turn counts into **PPMI** (positive pointwise mutual information: "do these appear together more than chance?") and compress each row with an **SVD** to `d` dimensions. Each row of the result is a dense embedding.

In [ ]:
total = C.sum()
p_ij = C / total
p_i = p_ij.sum(1, keepdims=True)
p_j = p_ij.sum(0, keepdims=True)
with np.errstate(divide="ignore"):
    ppmi = np.maximum(np.log(p_ij / (p_i * p_j)), 0)
ppmi[~np.isfinite(ppmi)] = 0

U, S, Vt = np.linalg.svd(ppmi)
d = 4
emb = U[:, :d] * np.sqrt(S[:d])           # (V, d) learned-from-data embeddings
print("embedding table:", emb.shape)

> 🧒 **In plain words:**
> This cell turns raw counts into a "surprise score". `p_ij` is the share of all counts for each pair, and `p_i`, `p_j` are the row and column shares. The `log(...)` line asks: does this pair meet more often than luck would predict? `np.maximum(..., 0)` throws away negative scores.
> Then `np.linalg.svd` squeezes each word's 17 numbers down to a short list. It is like summarising a long report into 4 key points. We keep the first `d = 4` of them, which gives `emb`.
> **Notice:** the printed shape `(17, 4)` means 17 words, each now described by just 4 numbers, the same kind of table as `E` earlier but built from the data.

## 5. Cosine similarity
Similarity of two vectors regardless of length: `cos(a, b) = a·b / (|a||b|)`; 1 = same direction, 0 = unrelated.

In [ ]:
def cos(a, b):
    return a @ b / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9)

def nearest(word, k=4):
    sims = [(w, cos(emb[idx[word]], emb[idx[w]])) for w in vocab if w != word]
    return sorted(sims, key=lambda t: -t[1])[:k]

for w in ["cat", "bread", "runs", "eats"]:
    print(f"{w:6s} ->", [(str(n), round(float(s), 2)) for n, s in nearest(w)])

> 🧒 **In plain words:**
> Think of each word's vector as an arrow. `cos` measures the angle between two arrows: `a @ b` multiplies matching numbers and adds them up, and dividing by the lengths (`np.linalg.norm`) removes the effect of arrow size. The tiny `1e-9` avoids dividing by zero.
> `nearest` scores a word against every other word and keeps the top 4 (`sorted` with `-t[1]` puts big scores first).
> **Notice:** the nearest words to "cat" are "dog", "mouse" and "horse" with score 1.0, which is the maximum. "bread" is closest to other foods, "runs" to other verbs, and "eats" to "likes" and "buys". Nobody told the program these groups.
> **Try it:** change `"cat"` to `"rice"` in the list. You should see other foods at the top.

Animals are near animals, foods near foods, verbs near verbs — **purely from how words are used**. A Transformer's embedding table learns the same kind of structure, just with gradient descent and a much bigger corpus.

In [ ]:
# 2-D picture via the first two principal directions
xy = emb[:, :2]
plt.figure(figsize=(6, 5))
for w in vocab:
    color = ("tab:blue" if w in animals else "tab:green" if w in foods
             else "tab:red" if w in a_verbs else "tab:orange" if w in people_verbs else "gray")
    plt.scatter(*xy[idx[w]], c=color)
    plt.annotate(w, xy[idx[w]], fontsize=9)
plt.title("embeddings (blue=animals, green=foods, red/orange=verbs)")
plt.show()

> 🧒 **In plain words:**
> We draw each word as a dot on a flat map using only its first two numbers, and write the word beside it. The long `color = ...` line is a chain of "if it is in this list, use this colour" choices.
> **Notice:** the plot has no printed text. Look for dots of the same colour sitting close together: animals in one clump, foods in another, and the verbs in their own spots. Dots of one group may overlap, because words used the same way get nearly identical coordinates.
> Think of it like a map of the school playground, where friends who play together stand together.

## 6. Embedding table size = a big chunk of the model
Parameters = `V × d_model`. A one-line check for GPT-2-small-like numbers:

In [ ]:
V_, d_ = 50_257, 768
print(f"{V_*d_:,} parameters just for token embeddings")

> 🧒 **In plain words:**
> A real model like GPT-2 small has 50,257 tokens and 768 numbers per token. The table has one number per row-and-column pair, so we multiply: `V_ * d_`. The `:,` in the print adds commas to the big number.
> **Notice:** about 38.6 million numbers, and that is only the vocabulary table. The model has to learn all of them.

## 7. Why multiply by √d_model?
The paper multiplies embeddings by `√d_model` before adding positional encoding (notebook 04), whose values are in `[-1, 1]`.
Embeddings are typically initialized with small variance (here `1/d_model`), so without scaling the position signal would drown the token signal.

In [ ]:
for d_ in [16, 64, 512]:
    E_ = np.random.default_rng(0).normal(0, 1/np.sqrt(d_), size=(1000, d_))   # variance 1/d
    norm = np.linalg.norm(E_, axis=1).mean()
    print(f"d_model={d_:4d}  |e| = {norm:.2f}   after *sqrt(d): {norm*np.sqrt(d_):.2f}")
print("\nPositional-encoding vector norm is ~ sqrt(d/2): comparable only after scaling.")

> 🧒 **In plain words:**
> For three sizes of `d_model` (16, 64, 512) we make 1000 random vectors, each with small numbers. `np.linalg.norm` is the length of a vector (like measuring an arrow with a ruler), and `.mean()` averages those 1000 lengths. Then we multiply by `sqrt(d)`, a square root.
> **Notice:** before scaling, the length is about 1 for every size, so the signal is tiny compared with what we add later. After scaling it grows: 3.93, 7.97 and 22.64. The scaled token vectors are big enough not to be drowned out by the position signal.
> Think of it like turning up the volume of a speaker before adding a second sound on top of it.
> **Try it:** add `1024` to the list of sizes. The scaled length should be even larger.

## Recap
- An embedding = a trainable `(V, d_model)` table; lookup ≡ one-hot @ table.
- Same token → same vector, **wherever it appears** (position comes next).
- Meaning emerges because similar contexts push vectors together.

➡ Next: **04 · Positional Encoding**.